# Grok-rl-05-pg-ppo

**Stage 05 — Policy Gradients → PPO**

## 概念
不学 Q，直接参数化 π(a|s;θ)，用回报优化似然。

演进：
1. **REINFORCE** — 蒙特卡洛策略梯度（高方差）
2. **Baseline / Advantage** — 减方差
3. **A2C** — 学习 V 作 baseline
4. **PPO** — clip 目标，现代主流稳定器

## 环境
同一 CartPole from scratch。


In [ ]:

import json, math, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.distributions import Categorical

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=0
np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu, device)

class CartPoleEnv:
    def __init__(self):
        self.g=9.8; self.mc=1.0; self.mp=0.1; self.tm=self.mc+self.mp
        self.l=0.5; self.pml=self.mp*self.l; self.f=10.0; self.tau=0.02
        self.th=12*math.pi/180; self.xt=2.4; self.reset()
    def reset(self):
        self.state=np.random.uniform(-0.05,0.05,size=4).astype(np.float32); self.t=0; return self.state.copy()
    def step(self,a):
        x,xd,th,thd=self.state
        force=self.f if a==1 else -self.f
        ct,st=math.cos(th),math.sin(th)
        temp=(force+self.pml*thd**2*st)/self.tm
        thacc=(self.g*st-ct*temp)/(self.l*(4/3-self.mp*ct**2/self.tm))
        xacc=temp-self.pml*thacc*ct/self.tm
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.state=np.array([x,xd,th,thd],np.float32); self.t+=1
        done=bool(abs(x)>self.xt or abs(th)>self.th or self.t>=500)
        return self.state.copy(), (0.0 if done else 1.0), done, {}

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.body=nn.Sequential(nn.Linear(4,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.pi=nn.Linear(64,2); self.v=nn.Linear(64,1)
    def forward(self,x):
        h=self.body(x); return self.pi(h), self.v(h).squeeze(-1)


In [ ]:

def run_reinforce(episodes=300, gamma=0.99, use_baseline=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    env=CartPoleEnv(); net=Policy().to(device); opt=torch.optim.Adam(net.parameters(), lr=3e-3)
    rets=[]
    for ep in range(episodes):
        s=env.reset(); logps=[]; rewards=[]; vals=[]; done=False
        while not done:
            st=torch.tensor(s,device=device)
            logits,v=net(st)
            dist=Categorical(logits=logits)
            a=dist.sample()
            logps.append(dist.log_prob(a)); vals.append(v); rewards.append(0.0)
            s,r,done,_=env.step(int(a.item())); rewards[-1]=r
        # returns
        G=0; Gs=[]
        for r in reversed(rewards):
            G=r+gamma*G; Gs.append(G)
        Gs=list(reversed(Gs))
        Gs_t=torch.tensor(Gs,device=device,dtype=torch.float32)
        logps_t=torch.stack(logps); vals_t=torch.stack(vals)
        if use_baseline:
            adv=Gs_t-vals_t.detach()
            loss=-(logps_t*adv).mean() + 0.5*F_mse(vals_t, Gs_t)
        else:
            # normalize returns for stability
            adv=(Gs_t-Gs_t.mean())/(Gs_t.std()+1e-8)
            loss=-(logps_t*adv).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        rets.append(sum(rewards))
    return np.array(rets)

def F_mse(a,b):
    return torch.mean((a-b)**2)

def run_ppo(episodes=400, gamma=0.99, lam=0.95, clip=0.2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    env=CartPoleEnv(); net=Policy().to(device); opt=torch.optim.Adam(net.parameters(), lr=3e-4)
    rets=[]; ep_ret=0; s=env.reset(); ep_count=0
    # collect by episodes for clarity
    while ep_count<episodes:
        # rollout one episode
        states=[]; acts=[]; logps=[]; rewards=[]; vals=[]; dones=[]
        done=False; ep_r=0
        while not done:
            st=torch.tensor(s,device=device)
            with torch.no_grad():
                logits,v=net(st)
                dist=Categorical(logits=logits)
                a=dist.sample()
            ns,r,done,_=env.step(int(a.item()))
            states.append(s); acts.append(int(a.item())); logps.append(float(dist.log_prob(a))); vals.append(float(v)); rewards.append(r); dones.append(done)
            s=ns; ep_r+=r
        # GAE
        vals.append(0.0)
        adv=np.zeros(len(rewards),dtype=np.float32); lastgaelam=0
        for t in reversed(range(len(rewards))):
            delta=rewards[t]+gamma*vals[t+1]*(0 if dones[t] else 1)-vals[t]
            lastgaelam=delta+gamma*lam*(0 if dones[t] else 1)*lastgaelam
            adv[t]=lastgaelam
        ret=adv+np.array(vals[:-1],dtype=np.float32)
        adv=(adv-adv.mean())/(adv.std()+1e-8)
        # ppo update
        st=torch.tensor(np.array(states),device=device,dtype=torch.float32)
        at=torch.tensor(acts,device=device)
        old_logp=torch.tensor(logps,device=device)
        adv_t=torch.tensor(adv,device=device); ret_t=torch.tensor(ret,device=device)
        for _ in range(4):
            logits,v=net(st)
            dist=Categorical(logits=logits)
            logp=dist.log_prob(at)
            ratio=torch.exp(logp-old_logp)
            surr1=ratio*adv_t
            surr2=torch.clamp(ratio,1-clip,1+clip)*adv_t
            loss=-torch.min(surr1,surr2).mean() + 0.5*F_mse(v, ret_t) - 0.01*dist.entropy().mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        rets.append(ep_r); ep_count+=1
        if done:
            s=env.reset()
    return np.array(rets)

t0=time.time()
ret_rf=run_reinforce(episodes=250, use_baseline=False, seed=1)
ret_bl=run_reinforce(episodes=250, use_baseline=True, seed=1)
ret_ppo=run_ppo(episodes=300, seed=1)
elapsed=time.time()-t0
print("REINFORCE", ret_rf[-30:].mean(), "baseline", ret_bl[-30:].mean(), "PPO", ret_ppo[-30:].mean())
print("elapsed", elapsed)


In [ ]:

def smooth(x,w=15):
    if len(x)<w: return x
    c=np.cumsum(np.insert(x.astype(float),0,0)); return (c[w:]-c[:-w])/w
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(smooth(ret_rf), label="REINFORCE")
ax.plot(smooth(ret_bl), label="REINFORCE+baseline")
ax.plot(smooth(ret_ppo), label="PPO")
ax.legend(); ax.set_title("Policy gradient family on CartPole"); ax.set_xlabel("episode"); ax.set_ylabel("return")
fig.tight_layout(); fig.savefig(OUT/"stage05_pg_ppo.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"05-pg-ppo",
  "title":"Grok-rl-05-pg-ppo",
  "metrics":{
    "reinforce_last30": float(ret_rf[-30:].mean()),
    "baseline_last30": float(ret_bl[-30:].mean()),
    "ppo_last30": float(ret_ppo[-30:].mean()),
  },
  "gpu": gpu,
  "elapsed_sec": elapsed,
  "concept": "optimize policy directly; PPO clips ratio for stable modern deep RL",
  "new_capability": "on-policy deep policy optimization (industry default: PPO)",
  "compare_to_previous": "Stage04 learned Q off-policy; Stage05 parameterizes pi and uses advantage/GAE",
}
assert payload["metrics"]["ppo_last30"] > 15
(OUT/"results_stage05.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE05_OK")
